# 2. [ASRJC/2025/PRELIM/P2/Q3]

A H2 Computing teacher has a collection of fill-in-the-blanks problems that needs to be managed using a MongoDB database.


## Task 1

The file `problems.txt` contains problems separated by an empty line.

Write program code to extract the data from the file and store them in a MongoDB collection named `problems` within a database named `computing`.

Note that the answers to the blanks in a problem should be stored as a list of strings, and the attempt count stored as an integer value.
<div style="text-align: right">[5]</div>


In [33]:
from pymongo import MongoClient

with open("resources/problems.txt") as f:
    data = [i.strip().split(": ",maxsplit=1) for i in f.readlines() if ": " in i]

to_insert = []
while True:
    counter = 0
    tmp = {}
    for key,val in data:
        if key == "Answers":
            val = val.split(",")
        if key == "Attempts":
            val = int(val)
        tmp[key] = val
        counter +=1
        if counter == 4:
            to_insert.append(tmp)
            counter = 0
            tmp = {}
    break
print(to_insert)

client = MongoClient()
database =client["computing"]
coll = database["problems"]
coll.drop()
coll.insert_many(to_insert)

[{'Topic': 'Recursion', 'Question': 'A recursive function is one that calls i___ within its definition. Every recursive function must have a b____ case to stop the recursion. Each recursive call should move the problem c_____ to the base case.', 'Answers': ['itself', 'base', 'closer'], 'Attempts': 23}, {'Topic': 'Data Structures', 'Question': 'A memory l___ occurs when dynamically allocated memory is not properly f___. This causes programs to consume increasing amounts of m___.', 'Answers': ['leak', 'freed', 'memory'], 'Attempts': 10}, {'Topic': 'Algorithms', 'Question': 'Binary search requires the array to be s___ beforehand and works by repeatedly h____ the search space.', 'Answers': ['sorted', 'halving'], 'Attempts': 2}, {'Topic': 'Data Structures', 'Question': 'In a queue, elements are added at the r___ and removed from the f___. This implements FIFO ordering.', 'Answers': ['rear', 'front'], 'Attempts': 12}, {'Topic': 'Computer Networking', 'Question': 'A L___ Area Network connects

InsertManyResult([ObjectId('6a4306ec1e9854db9162352a'), ObjectId('6a4306ec1e9854db9162352b'), ObjectId('6a4306ec1e9854db9162352c'), ObjectId('6a4306ec1e9854db9162352d'), ObjectId('6a4306ec1e9854db9162352e'), ObjectId('6a4306ec1e9854db9162352f'), ObjectId('6a4306ec1e9854db91623530'), ObjectId('6a4306ec1e9854db91623531'), ObjectId('6a4306ec1e9854db91623532'), ObjectId('6a4306ec1e9854db91623533'), ObjectId('6a4306ec1e9854db91623534'), ObjectId('6a4306ec1e9854db91623535'), ObjectId('6a4306ec1e9854db91623536'), ObjectId('6a4306ec1e9854db91623537'), ObjectId('6a4306ec1e9854db91623538'), ObjectId('6a4306ec1e9854db91623539'), ObjectId('6a4306ec1e9854db9162353a'), ObjectId('6a4306ec1e9854db9162353b'), ObjectId('6a4306ec1e9854db9162353c'), ObjectId('6a4306ec1e9854db9162353d'), ObjectId('6a4306ec1e9854db9162353e'), ObjectId('6a4306ec1e9854db9162353f'), ObjectId('6a4306ec1e9854db91623540'), ObjectId('6a4306ec1e9854db91623541'), ObjectId('6a4306ec1e9854db91623542'), ObjectId('6a4306ec1e9854db916235

## Task 2

Write MongoDB query logic to extract problems that meet both of the following criteria:

 * the topic is either `‘Data Structures’` or `‘Algorithms’`
 * the number of attempts for the problems is between 5 and 10 (both inclusive).

Display the question texts of these matching problems. <div style="text-align: right">[3]</div>


In [42]:
from pymongo import MongoClient

client = MongoClient()
database = client["computing"]
coll = database["problems"]

doc = coll.find({"$or":[{"Topic":"Data Structures"},{"Topic":"Algorithms"}],"Attempts":{"$gte":5,"$lte":10}})
print(*list(doc),sep="\n")

{'_id': ObjectId('6a4306ec1e9854db9162352b'), 'Topic': 'Data Structures', 'Question': 'A memory l___ occurs when dynamically allocated memory is not properly f___. This causes programs to consume increasing amounts of m___.', 'Answers': ['leak', 'freed', 'memory'], 'Attempts': 10}
{'_id': ObjectId('6a4306ec1e9854db91623530'), 'Topic': 'Data Structures', 'Question': 'In a binary search tree, all values in the left subtree are s___ than the root, and all values in the right subtree are l___.', 'Answers': ['smaller', 'larger'], 'Attempts': 9}
{'_id': ObjectId('6a4306ec1e9854db91623532'), 'Topic': 'Algorithms', 'Question': 'Bubble sort compares a___ elements and s___ them if they are in wrong order.', 'Answers': ['adjacent', 'swaps'], 'Attempts': 6}
{'_id': ObjectId('6a4306ec1e9854db91623534'), 'Topic': 'Algorithms', 'Question': 'Merge sort uses d___ and c___ strategy, recursively splitting arrays then m___ sorted subarrays back together.', 'Answers': ['divide', 'conquer', 'merging'], 'Att

## Task 3

Since the topic `‘Socket Programming’` will no longer be there in the revised H2 Computing syllabus, the teacher wants to update the collection by removing all problems tagged with that topic.

Write MongoDB logic to perform this update and execute it.
<div style="text-align: right">[2]</div>


In [44]:
from pymongo import MongoClient

client = MongoClient()
database = client["computing"]
coll = database["problems"]

coll.delete_many({"Topic":"Socket Programming"})

DeleteResult({'n': 3, 'ok': 1.0}, acknowledged=True)

## Task 4

Write program code that:

 * randomly selects a problem from the MongoDB collection
 * displays the topic and the question text of the selected problem
 * prompts the student to enter an answer for each blank in the question, reminding him to include the first letter in his response
 * checks the student’s answers against the expected answers (case-insensitive)
 * outputs summative results (e.g. “You got 3 out of 5 answers correct\!”), without revealing what the correct answers are
 * increments the attempt count for that problem by 1 in the database.

If the student’s answers to the problem are not all correct, prompt the student to enter all the answers again for the same problem.

This continues until the student answers the entire problem correctly.

Test your code appropriately.
<div style="text-align: right">[7]</div>


In [49]:
from pymongo import MongoClient

client = MongoClient()
db = client["computing"]
coll = db["problems"]

qn = list(coll.aggregate([{"$sample":{"size":1}}]))[0]
id = qn["_id"]
print("Topic:", qn["Topic"])
print(qn["Question"])
print()
print("Remember to include the first letter in your response.")
print()
answers = qn["Answers"]
while True:
    score = 0
    for i in range(len(answers)):
        ans = input(f"Enter your answer for blank {i}: ")
        if ans.lower() == answers[i].lower():
            score += 1
    print(f"You got {score} out of {len(answers)} correct!")
    coll.update_one({"_id":id},{"$inc":{"Attempts":1}})
    if score == len(answers):
        break


    

Topic: Computer Networking
A V___ P___ N___ creates a secure tunnel over a public network. Client-server architecture has clients making r___ to servers that provide r___.

Remember to include the first letter in your response.

You got 5 out of 5 correct!
